In [6]:
%pip install scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/9f/c4/0ab22726a04ede56f689476b760f98f8f46607caecff993017ac1b64aa5d/scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/a2/84/dc08d77fbf3d87d3ee27f6a0c6dcce1de5829a64f2eae85a0ecc1f0daa73/scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Obtaining dependency information for joblib>=1.3.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Obtaining dependency information for threadpoolctl>=3.2.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define Features (X) and Target (y)
X = df[['LastPurchaseDaysAgo', 'TotalPurchases', 'TotalSpend', 'R', 'F', 'M']]
y = df['Churned']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict and Evaluate
predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print(classification_report(y_test, predictions))

Accuracy: 0.82
              precision    recall  f1-score   support

           0       0.83      0.98      0.90       165
           1       0.43      0.09      0.14        35

    accuracy                           0.82       200
   macro avg       0.63      0.53      0.52       200
weighted avg       0.76      0.82      0.77       200



In [4]:
# 2. Calculate RFM Scores using Quantiles
quantiles = df[['LastPurchaseDaysAgo', 'TotalPurchases', 'TotalSpend']].quantile(q=[0.25, 0.5, 0.75]).to_dict()

def r_score(x, p, d):
    if x <= d[p][0.25]: return 4 # Bought recently = Good (4)
    elif x <= d[p][0.50]: return 3
    elif x <= d[p][0.75]: return 2
    else: return 1

def fm_score(x, p, d):
    if x <= d[p][0.25]: return 1 # Spent little/Bought rarely = Bad (1)
    elif x <= d[p][0.50]: return 2
    elif x <= d[p][0.75]: return 3
    else: return 4

df['R'] = df['LastPurchaseDaysAgo'].apply(r_score, args=('LastPurchaseDaysAgo', quantiles,))
df['F'] = df['TotalPurchases'].apply(fm_score, args=('TotalPurchases', quantiles,))
df['M'] = df['TotalSpend'].apply(fm_score, args=('TotalSpend', quantiles,))

# Combine into one RFM Score string
df['RFM_Segment'] = df['R'].map(str) + df['F'].map(str) + df['M'].map(str)
df.head()

,CustomerID,LastPurchaseDaysAgo,TotalPurchases,TotalSpend,Churned,R,F,M,RFM_Segment
0,1,103,44,2137.518588,0,3,4,2,342
1,2,349,2,1420.297596,0,1,1,2,112
2,3,271,13,2982.134127,0,1,1,3,113
3,4,107,40,4566.198561,0,3,4,4,344
4,5,72,2,1092.777836,0,4,1,1,411


In [3]:
import pandas as pd
import numpy as np
from datetime import timedelta

# 1. Generate Dummy Data
np.random.seed(42)
n_customers = 1000

data = {
    'CustomerID': range(1, n_customers + 1),
    'LastPurchaseDaysAgo': np.random.randint(1, 365, n_customers),
    'TotalPurchases': np.random.randint(1, 50, n_customers),
    'TotalSpend': np.random.uniform(50, 5000, n_customers),
    'Churned': np.random.choice([0, 1], p=[0.8, 0.2], size=n_customers) # 20% churn rate
}

df = pd.DataFrame(data)
df.head()

,CustomerID,LastPurchaseDaysAgo,TotalPurchases,TotalSpend,Churned
0,1,103,44,2137.518588,0
1,2,349,2,1420.297596,0
2,3,271,13,2982.134127,0
3,4,107,40,4566.198561,0
4,5,72,2,1092.777836,0
